# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading, overviewing, and processing the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library according to the [Croissant schema](https://mlcommons.org/croissant/).

### Dataset Source
The dataset source (Croissant schema) is provided by this URL and can be used directly with `mlcroissant`.

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load the metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset title: {metadata.name}\n")
print(f"Dataset description: {metadata.description}\n")
print(f"Dataset Croissant identifier (@id): {metadata.id}\n")
print(f"Croissant version: {getattr(metadata, 'conforms_to', getattr(metadata, 'conformsTo', None))}")

## 2. Data Overview
Review available record sets and their fields. All entities are referred to by their Croissant `@id` for robust referencing.

This dataset may include multiple record sets (tables, objects, etc.). We'll enumerate all available record sets and display their associated field and column `@id`s.

In [ ]:
# List all record sets in the dataset
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record sets.\n")
for i, record_set in enumerate(record_sets):
    print(f"[{i}] Record set name: {record_set.name}")
    print(f"    @id: {record_set.id}")
    print(f"    Description: {record_set.description}")
    print(f"    Fields (@id):")
    for field in record_set.fields:
        print(f"      - {field.name}: {field.id}")
    print(f"    Columns (@id):")
    for col in record_set.columns:
        print(f"      - {col.name}: {col.id}")
    print()

## 3. Data Extraction
Load records from a specific record set into a DataFrame for analysis. Use explicit Croissant `@id` strings for selection.

In [ ]:
# If there are multiple record sets, pick the main protein table for analysis
# You can inspect record_sets above and select the correct @id
if record_sets:
    # Choose the first record set as main table (replace with correct index or @id if different)
    main_record_set = record_sets[0]
    main_record_set_id = main_record_set.id
    print(f"Using record set '@id': {main_record_set_id}\n")
    # List all available record set IDs for further selection
    print("Available record_set @ids:")
    for rs in record_sets:
        print("  -", rs.id)

    # Load all records from the chosen record set
    df = pd.DataFrame(list(dataset.records(record_set=main_record_set_id)))
    print(f"Sample columns ({len(df.columns)}):\n", df.columns.tolist())
    display(df.head())
else:
    print("No record sets found in the dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records, normalizing numeric fields, grouping and summarizing the data.

**All column references are by their Croissant `@id`.**

In [ ]:
if not df.empty:
    # Enumerate available numeric columns (referenced by @id)
    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
    print(f"Numeric columns: {numeric_cols}")
    # Choose a representative numeric field (replace with correct @id if available)
    if numeric_cols:
        numeric_field_id = numeric_cols[0]
        print(f"Using numeric field: {numeric_field_id}")

        # Example filter: records with value above mean
        threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > mean ({threshold:.2f}): {filtered_df.shape[0]} rows")
        display(filtered_df.head())

        # Normalization
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized '{numeric_field_id}':")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Attempt grouping by a candidate field, e.g., 'accession' if available
        group_field_candidates = [col for col in df.columns if col.lower().startswith('accession') or 'group' in col.lower()]
        group_field = group_field_candidates[0] if group_field_candidates else None

        if group_field:
            print(f"Grouping by field: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print("Grouped data sample:")
            display(grouped_df.head())
        else:
            print("No grouping field found.")
    else:
        print("No numeric columns in the main record set.")
else:
    print("No data to analyze.")

## 5. Visualization
Visualize one or more numeric fields, using Croissant `@id` for all references.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not df.empty and 'numeric_field_id' in locals():
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If grouping field exists, plot group mean
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(12,5))
        group_values = df.groupby(group_field)[numeric_field_id].mean().sort_values(ascending=False)
        group_values.plot(kind='bar')
        plt.title(f"Mean {numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.tight_layout()
        plt.show()

## 6. Conclusion
We have loaded and explored the FAIR^2 dataset on mass spectrometry analysis of extracellular vesicles. Using the Croissant schema and explicit referencing by `@id`, we overviewed available record sets, extracted tabular data, performed basic EDA, filtered and normalized fields, and visualized the data. Further biological or domain-specific analysis could expand upon these foundations.

**For reproducibility, always reference fields, record sets, and columns by their Croissant `@id`.**